<a href="https://colab.research.google.com/github/asoknaren/AIPlayground/blob/main/001_tokenizer_microsoft_deberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import time

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-xsmall"
TOKENIZER_NAME = "microsoft/deberta-base"
TARGET_WORDS = 200

In [ ]:
def build_prompt() -> str:
    return (
        "Write a sincere professional apology email in about 200 words. "
        "Context: I missed an important client deadline due to poor planning. "
        "The email should acknowledge responsibility, explain briefly without excuses, "
        "and include a clear corrective action plan and request to rebuild trust."
    )


In [ ]:
prompt = build_prompt()

In [ ]:
def initialize_model_and_tokenizer(model_name: str):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    model = AutoModel.from_pretrained(model_name, dtype=torch.bfloat16)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    if device == "cpu":
        cpu_threads = max(1, (torch.get_num_threads() or 1) - 1)
        torch.set_num_threads(cpu_threads)
        print(f"Using CPU threads: {cpu_threads}")

    return tokenizer, model, device

In [ ]:
tokenizer, model, device = initialize_model_and_tokenizer(MODEL_NAME)

## Show Tokens Helper Function

To better visualize the tokenization process, here's a helper function that displays tokens with different background colors.

In [ ]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence):
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [ ]:
show_tokens(prompt)

In [ ]:
def tokenize_and_display_prompt_info(tokenizer, prompt: str, device: str):
    # Use the tokenizer directly for encoding, as chat_template is not applicable for this model
    model_inputs = tokenizer(prompt, return_tensors="pt")
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    input_ids = model_inputs["input_ids"]
    token_count = input_ids.shape[1]

    print(f"Prompt token count: {token_count}")
    print(f"Equivalent tokenized words: {tokenizer.decode(input_ids[0], skip_special_tokens=True)}")

    print("\nToken to Word Mapping:")
    for token_id in input_ids[0]:
        token_word = tokenizer.decode(token_id)
        print(f"Token ID: {token_id.item()}, Word: '{token_word}'")

    return model_inputs, token_count

In [ ]:
model_inputs, token_count = tokenize_and_display_prompt_info(tokenizer, prompt, device)

The `microsoft/deberta-v3-xsmall` model is an **encoder-only** transformer model. Unlike models like `phi-3-mini` (which is a decoder-only model used for text generation), encoder-only models are primarily designed for understanding and encoding text into rich numerical representations (embeddings).

Here's what you can typically do with an encoder-only model like DeBERTa:

*   **Text Classification**: Classify text into categories (e.g., sentiment analysis, spam detection, topic classification).
*   **Named Entity Recognition (NER)**: Identify and classify named entities in text (e.g., person names, organizations, locations).
*   **Question Answering (Extractive)**: Given a question and a context paragraph, find the exact span of text in the context that answers the question.
*   **Semantic Search**: Find documents or passages that are semantically similar to a query.
*   **Text Similarity/Paraphrase Detection**: Determine how similar two pieces of text are.

The model's output provides contextual embeddings for each token in the input. These embeddings capture the meaning of words in the context of the surrounding words. You can then use these embeddings as features for a downstream task (often by adding a simple classification head on top of the encoder).

In [ ]:
# Get the model's output (contextual embeddings)
with torch.no_grad():
    model_output = model(**model_inputs)

# The last_hidden_state contains the contextual embeddings for each token
# The shape is (batch_size, sequence_length, hidden_size)
last_hidden_state = model_output.last_hidden_state

print(f"Shape of last_hidden_state: {last_hidden_state.shape}")

# To get a single sentence embedding, you can typically average the token embeddings
# or use the embedding of the [CLS] token (the first token).

# Option 1: Average pooling (mean of all token embeddings)
sentence_embedding_avg = torch.mean(last_hidden_state, dim=1)
print(f"Shape of averaged sentence embedding: {sentence_embedding_avg.shape}")

# Option 2: Using the [CLS] token embedding (first token)
# This is common for classification tasks where the [CLS] token is specifically trained to represent the whole sequence.
sentence_embedding_cls = last_hidden_state[:, 0, :]
print(f"Shape of [CLS] token embedding: {sentence_embedding_cls.shape}")

print("\nFirst 5 values of the [CLS] token embedding:")
print(sentence_embedding_cls[0, :5])

print("\nThese embeddings can then be used as input features for a task-specific head (e.g., a linear layer for classification).")